# Track 1 — LoRA Fine-Tuning Orchestrator

Thin orchestrator only. All real logic lives in `track1_finetune/scripts/*.py`
(agent-editable, ordinary `.py` modules). Cells below just **sync code**, **install
deps**, and **call into those scripts**.

**Before running the sync cell:** after any local agent edit you MUST commit and push
(`git add -A && git commit -m ... && git push`) so the remote kernel pulls your
latest code. Re-running the sync cell picks up new edits.

In [ ]:
!git clone https://github.com/DevaNandanJS/Benchmarking-LLM-fine-tuning-vs-training-from-scratch-using-the-same-dataset.git llm_task 2>/dev/null || (cd llm_task && git pull)
%cd llm_task

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected - connect a Colab GPU kernel first"
props = torch.cuda.get_device_properties(0)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(props.total_memory / 1e9, 2))
print("torch:", torch.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Phase 0 — Environment verification

The script below, `env_check.py`, is the Phase 0 environment verification — it also
serves as the "test local edit" that the sync cell is verified against (Phase 0 DoD).
Results land under `track1_finetune/logs/` and are pulled back into the repo by the user.

In [ ]:
!python track1_finetune/scripts/env_check.py

In [ ]:
!pip freeze > track1_finetune/environment.txt
print("Wrote track1_finetune/environment.txt - commit this back to the repo (reproducibility lock).")

## Phase 1 — Data Extraction & Cleaning

Runs `extract_text.py` which uses **pdfplumber** (primary) / **pypdf** (per-page fallback).

Outputs in `data/extracted/`:
- `document_clean.txt` — final cleaned text  
- `stats.json` — char / word / proxy-token counts  
- `extraction_manifest.json` — per-page extractor choice + quality scores  
- `hyphen_join_decisions.txt` — audit log of every line-break hyphen decision  
- `raw_pages/` — pre-clean text from both extractors, per page  

**After running:** open `data/extracted/document_clean.txt` and spot-check a few
random sections. Check `extraction_manifest.json` for any pages with `"garbled_flag": true`.

In [ ]:
!python track1_finetune/scripts/extract_text.py

In [ ]:
# Quick sanity check: first & last 300 chars + stats summary
with open('data/extracted/document_clean.txt', encoding='utf-8') as f:
    text = f.read()
print('--- FIRST 300 CHARS ---')
print(text[:300])
print('\n--- LAST 300 CHARS ---')
print(text[-300:])

import json
stats = json.load(open('data/extracted/stats.json'))
print('\n--- STATS ---')
for k, v in stats.items():
    if k != 'note':
        print(f'  {k}: {v}')

# Show any garbled pages
manifest = json.load(open('data/extracted/extraction_manifest.json'))
garbled = [m for m in manifest if m['garbled_flag']]
if garbled:
    print(f'\n⚠ {len(garbled)} page(s) flagged as garbled:')
    for m in garbled:
        print(f"  page {m['page']:3d}  chosen={m['extractor_chosen']}  reason={m['reason']}")
        print(f"          non_ascii={m['final_scores']['non_ascii_ratio']:.1%}  "
              f"non_dict={m['final_scores']['non_dict_word_ratio']:.1%}")
else:
    print('\n✓ No pages flagged as garbled.')

## Phase 2 — Base Model & Tokenizer Selection

Runs `select_model.py` which:
1. Loads `HuggingFaceTB/SmolLM2-135M` tokenizer + model
2. Confirms/sets the pad token and logs the decision
3. Tokenizes `document_clean.txt` with the real tokenizer and updates `stats.json`
4. Extracts all `model.named_modules()` names → `configs/model_architecture.json`
   (Phase 4 reads this to set LoRA `target_modules` from verified names)
5. Runs fp16 memory math and records the quantization decision (no QLoRA needed)
6. Dumps `configs/run_phase2.json`

**After running:** check that:
- `data/extracted/stats.json` now has `exact_token_count_smollm2_135m`
- `configs/model_architecture.json` exists and lists `q_proj`/`v_proj`/etc.
- `configs/run_phase2.json` exists with `quantization_needed: false`
- All DoD assertions printed `✓`

In [ ]:
!python track1_finetune/scripts/select_model.py

In [ ]:
# Phase 2 post-run inspection
import json

# 1. Updated token counts
stats = json.load(open('data/extracted/stats.json'))
print('--- TOKEN COUNTS ---')
print(f"  proxy_token_count_gpt2_tiktoken : {stats.get('proxy_token_count_gpt2_tiktoken')}")
print(f"  exact_token_count_smollm2_135m  : {stats.get('exact_token_count_smollm2_135m')}")

# 2. Run config summary
cfg = json.load(open('track1_finetune/configs/run_phase2.json'))
print('\n--- RUN CONFIG SUMMARY ---')
for k in ('model_name', 'total_params', 'pad_token', 'quantization_needed',
          'fp16_weight_footprint_mb', 'vram_utilisation_weights_only_pct'):
    print(f"  {k}: {cfg.get(k)}")

# 3. Architecture spot-check — attention projection layers for Phase 4
arch = json.load(open('track1_finetune/configs/model_architecture.json'))
attn_kw = ('q_proj', 'k_proj', 'v_proj', 'o_proj', 'c_attn')
attn = [n for n in arch['all_module_names'] if any(k in n for k in attn_kw)]
print(f"\n--- ATTENTION MODULES ({len(attn)} found — first 10) ---")
for name in attn[:10]:
    print(f"  {name}")


## Phase 3 - Dataset Construction (Chunking & Splitting)

Runs `build_dataset.py` which:
1. Loads SmolLM2-135M tokenizer and tokenizes `document_clean.txt`
2. Splits tokens into train (85%) / val (15%) by **contiguous holdout** (last 15% of token sequence)
3. Chunks train with stride=128 (50% overlap) for dense training samples
4. Chunks val with stride=256 (non-overlapping) for independent validation spans
5. Saves pure tensor dicts to `data/processed/track1_train.pt` and `track1_val.pt`
6. Writes `data/processed/dataset_stats.json` and `configs/split_strategy.md`

**After running:** check `dataset_stats.json` for exact chunk counts,
then spot-check a decoded sample chunk to confirm it reads like real document text.

In [ ]:
!python track1_finetune/scripts/build_dataset.py

In [ ]:
# Phase 3 post-run inspection
import json, torch

# 1. Dataset stats
stats = json.load(open('data/processed/dataset_stats.json'))
print('--- DATASET STATS ---')
for k, v in stats.items():
    if k not in ('model_name', 'timestamp', 'note'):
        print(f'  {k}: {v}')

# 2. Tensor shapes (weights_only=True is clean -- pure tensor dict, no custom classes)
train_data = torch.load('data/processed/track1_train.pt', weights_only=True)
val_data   = torch.load('data/processed/track1_val.pt',   weights_only=True)
print('\n--- TENSOR SHAPES ---')
print('  train input_ids:', tuple(train_data['input_ids'].shape))
print('  val   input_ids:', tuple(val_data['input_ids'].shape))

# 3. Spot-check: decode first train and first val chunk
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('HuggingFaceTB/SmolLM2-135M')
print('\n--- TRAIN CHUNK[0] (first 80 tokens decoded) ---')
print(tok.decode(train_data['input_ids'][0][:80]))
print('\n--- VAL CHUNK[0] (first 80 tokens decoded) ---')
print(tok.decode(val_data['input_ids'][0][:80]))
